# `AA Workshop 12` — Coding Challenge

Complete the tasks below to practice collaborative filtering techniques from `W12_Recommender_Systems.ipynb`.

Guidelines:
- Work in order. Run each cell after editing with Shift+Enter.
- Keep answers short; focus on making things work.
- If a step fails, read the error and fix it.

By the end you will have exercised:
- implementing item- and user-based approaches to predict ratings
- generating recommendations for a specific user

## Task 1 - Predict a specific rating

Let's apply what we learned about collaborative filtering. We will use the same datasets as in the workshop notebook, i.e. `ratings.csv` and `movies.csv` from https://grouplens.org/datasets/movielens/. Again, we only want to consider movies with five or more ratings. The user with `userId = 15` has not yet rated the movie named _Beauty and the Beast (1991)_. First, check out some movies the user has rated with the highest score (5). Then, apply and compare item-item and user-user approaches using Pearson correlation and Cosine similarity as similarity measures to predict whether the user will likely enjoy or dislike this movie _Beauty and the Beast (1991)_. Given the users most and least favorite movies, did you expect the predicted rating for _Beauty and the Beast (1991)_?

In [1]:
import numpy as np
import pandas as pd

ratings = pd.read_csv('../data/ratings.csv')
ratings = ratings.groupby('movieId').filter(lambda x: len(x) >= 6)
movies = pd.read_csv('../data/movies.csv')
display(ratings.head(2))
display(movies.head(2))

def get_movie_title(movie_id):
    return movies[movies['movieId'] == movie_id]['title'].values[0]
def get_movie_id(movie_title):
    return movies[movies['title'] == movie_title]['movieId'].values[0]

assert get_movie_id('Toy Story (1995)') == 1
assert get_movie_title(1) == 'Toy Story (1995)'

,userId,movieId,rating,timestamp
0,1,31,2.5,1260759144
1,1,1029,3.0,1260759179


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy


In [2]:
user_id = 15
user_ratings = ratings[ratings['userId'] == user_id]
print(f"Most liked movies of user {user_id}:")
print("\n".join(user_ratings.loc[user_ratings['rating'] == 5.0, 'movieId'].head(10).apply(get_movie_title).tolist()))

Most liked movies of user 15:
Seven (a.k.a. Se7en) (1995)
Usual Suspects, The (1995)
Antonia's Line (Antonia) (1995)
Taxi Driver (1976)
Amateur (1994)
Hoop Dreams (1994)
Star Wars: Episode IV - A New Hope (1977)
Léon: The Professional (a.k.a. The Professional) (Léon) (1994)
Pulp Fiction (1994)
Three Colors: Red (Trois couleurs: Rouge) (1994)


In [3]:
X = ratings.pivot(index='userId', columns='movieId', values='rating').fillna(0).to_numpy()
print(f'Number of ratings: {X.size - np.sum(X == 0)} (out of {X.size} total entries)')
print(X.shape[0], 'users and', X.shape[1], 'movies')

Number of ratings: 88087 (out of 2079429 total entries)
671 users and 3099 movies


In [4]:
user_means = np.array([X[i,X[i,:]!=0].mean() for i in range(X.shape[0])])
movie_means = np.array([X[X[:,i]!=0,i].mean() for i in range(X.shape[1])])

User-user approach

In [5]:
def all_pearson(X, user_means):
    X_norm = (X - user_means[:,None])*(X != 0)
    X_col_norm = (X_norm**2) @ (X_norm != 0).T
    return (X_norm @ X_norm.T)/(np.sqrt(X_col_norm*X_col_norm.T)+1e-12)
W_pearson_user = all_pearson(X, user_means)

def all_cosine(X):
    x_norm = np.sqrt((X**2).sum(axis=1))
    return (X @ X.T) / np.outer(x_norm, x_norm)
W_cosine_user = all_cosine(X)

def predict_user_user(X, W, user_means, i):
    """ Return prediction of X_(ij). """
    return user_means[i] + (np.sum((X - user_means[:,None]) * (X != 0) * W[i,:,None], axis=0) / 
                            (np.sum((X != 0) * np.abs(W[i,:,None]), axis=0) + 1e-12))


pearson_pred = predict_user_user(X, W_pearson_user, user_means, user_id-1)[get_movie_id('Beauty and the Beast (1991)')-1]
cosine_pred = predict_user_user(X, W_cosine_user, user_means, user_id-1)[get_movie_id('Beauty and the Beast (1991)')-1]
print(f'Predicted rating for user {user_id} (mean rating {user_means[user_id-1]:.2f}) for "Beauty and the Beast (1991)":')
print(f'  using Pearson similarity: {pearson_pred:.4f}')
print(f'  using Cosine similarity:  {cosine_pred:.4f}')

Predicted rating for user 15 (mean rating 2.65) for "Beauty and the Beast (1991)":
  using Pearson similarity: 3.2208
  using Cosine similarity:  3.0832


Item-item approach

In [6]:
W_pearson_item = all_pearson(X.T, movie_means)
W_cosine_item = all_cosine(X.T)

def predict_item_item(X, W, item_means, i):
    return predict_user_user(X.T, W, item_means, i)


print(f'Predicted rating for user {user_id} (mean rating {user_means[user_id-1]:.2f}) for "Beauty and the Beast (1991)":')
print(f'  using Pearson similarity: {predict_item_item(X, W_pearson_item, movie_means, get_movie_id("Beauty and the Beast (1991)")-1)[user_id-1]:.4f}')
print(f'  using Cosine similarity:  {predict_item_item(X, W_cosine_item, movie_means, get_movie_id("Beauty and the Beast (1991)")-1)[user_id-1]:.4f}')

Predicted rating for user 15 (mean rating 2.65) for "Beauty and the Beast (1991)":
  using Pearson similarity: 4.3318
  using Cosine similarity:  3.1826


## Task 2 - Recommend five movies

Task 1 should have told you that _Beauty and the Beast (1991)_ is likely not the best recommendation to give to the user with `userId = 15`. Again, apply and compare item-item and user-user approaches using Pearson correlation and Cosine similarity as similarity measures to recommend the five movies with the highest predicted rating.

In [7]:
n = 5
# user-user recommendations
for name, W in zip(['Pearson', 'Cosine'], [W_pearson_user, W_cosine_user]):
    pred = predict_user_user(X, W, user_means, user_id-1)
    top_n_indices = np.argsort(pred)[-n:][::-1]
    movie_id_map = ratings.pivot(index='userId', columns='movieId', values='rating').columns
    
    print(f'\nTop {n} recommendations for user {user_id} using {name} similarity:')
    for num, i in enumerate(top_n_indices, 1):
        movie_id = movie_id_map[i]
        title = get_movie_title(movie_id)
        print(f'{num}. {title}: predicted rating: {pred[i]:.2f}')


Top 5 recommendations for user 15 using Pearson similarity:
1. It Follows (2014): predicted rating: 4.09
2. In the Bedroom (2001): predicted rating: 3.97
3. Battle of Algiers, The (La battaglia di Algeri) (1966): predicted rating: 3.90
4. Anne Frank Remembered (1995): predicted rating: 3.84
5. Amateur (1994): predicted rating: 3.81

Top 5 recommendations for user 15 using Cosine similarity:
1. Anne Frank Remembered (1995): predicted rating: 3.86
2. Yojimbo (1961): predicted rating: 3.81
3. Amateur (1994): predicted rating: 3.77
4. Trip to the Moon, A (Voyage dans la lune, Le) (1902): predicted rating: 3.74
5. Battle of Algiers, The (La battaglia di Algeri) (1966): predicted rating: 3.74


In [8]:
n = 5
movie_id_map = ratings.pivot(index='userId', columns='movieId', values='rating').columns

for name, W in zip(['Pearson', 'Cosine'], [W_pearson_item, W_cosine_item]):
    user_ratings = X[user_id - 1, :]
    numerator = user_ratings @ W
    denominator = (user_ratings != 0) @ np.abs(W)
    pred = numerator / (denominator + 1e-12)
    rated_indices = np.where(X[user_id - 1] > 0)[0]
    pred[rated_indices] = -np.inf
    top_n_indices = np.argsort(pred)[-n:][::-1]
    print(f'\nTop {n} recommendations for user {user_id} using item-item {name} similarity:')
    for num, i in enumerate(top_n_indices, 1):
        movie_id = movie_id_map[i]
        title = get_movie_title(movie_id)
        print(f'{num}. {title}: predicted rating: {pred[i]:.2f}')


Top 5 recommendations for user 15 using item-item Pearson similarity:
1. How Green Was My Valley (1941): predicted rating: 2.03
2. Revolutionary Road (2008): predicted rating: 1.99
3. Flushed Away (2006): predicted rating: 1.98
4. All the King's Men (1949): predicted rating: 1.97
5. Beverly Hills Ninja (1997): predicted rating: 1.93

Top 5 recommendations for user 15 using item-item Cosine similarity:
1. Safe (1995): predicted rating: 3.21
2. Strawberry and Chocolate (Fresa y chocolate) (1993): predicted rating: 3.19
3. Grand Hotel (1932): predicted rating: 3.10
4. Burnt by the Sun (Utomlyonnye solntsem) (1994): predicted rating: 3.08
5. Shall We Dance (1937): predicted rating: 3.07
